# Fake News Detection Pipeline

This notebook demonstrates a complete ML lifecycle: EDA, Data Preprocessing, Feature Engineering, Model Training, Evaluation, and Ensembling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import sys
sys.path.append('..')
from src.preprocess import get_train_test_splits
from src.models import BiLSTM, SimpleTokenizer, get_traditional_models
from src.ensemble import save_ensemble_config

os.makedirs('../model_pipeline', exist_ok=True)


## 1. Data Loading & Preprocessing

In [ ]:
train_df, valid_df, test_df = get_train_test_splits('../train.tsv', '../valid.tsv', '../test.tsv')
print(f"Train size: {len(train_df)}")
print(f"Valid size: {len(valid_df)}")


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution Plot
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=train_df)
plt.title('Class Distribution (0 = Fake, 1 = Real)')
plt.show()

# Text Length Distribution
train_df['text_len'] = train_df['text'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10, 5))
sns.histplot(data=train_df, x='text_len', hue='target', bins=50, kde=True)
plt.title('Text Length Distribution by Target')
plt.xlim(0, 100)
plt.show()


## 3. Classical Model Training

In [ ]:
X_train = train_df['text'].tolist()
y_train = train_df['target'].tolist()
X_test = test_df['text'].tolist()
y_test = test_df['target'].tolist()

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
joblib.dump(tfidf, '../model_pipeline/tfidf_vectorizer.pkl')

models = get_traditional_models()
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_tfidf, y_train)
    score = model.score(X_test_tfidf, y_test)
    print(f"{name} Accuracy: {score:.4f}")
    if name == 'logistic':
        joblib.dump(model, '../model_pipeline/logistic_model.pkl')
    else:
        joblib.dump(model, f'../model_pipeline/{name}_model.pkl')


## 4. LSTM Training

In [ ]:
tokenizer = SimpleTokenizer(max_vocab_size=5000, max_len=128)
tokenizer.fit(X_train)
joblib.dump(tokenizer, '../model_pipeline/tokenizer.pkl')

lstm_model = BiLSTM(vocab_size=len(tokenizer))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

# Training block (1 epoch for demo)
lstm_model.train()
for i in range(0, len(X_train), 64):
    batch_texts = X_train[i:i+64]
    batch_y = torch.tensor(y_train[i:i+64], dtype=torch.float32)
    encoded = [tokenizer.pad(tokenizer.encode(t)) for t in batch_texts]
    batch_x = torch.tensor(encoded, dtype=torch.long)
    
    optimizer.zero_grad()
    preds = lstm_model(batch_x)
    loss = criterion(preds, batch_y)
    loss.backward()
    optimizer.step()

torch.save(lstm_model.state_dict(), '../model_pipeline/lstm_model.pt')
print("LSTM trained and saved!")


## 5. DistilBERT Fine-Tuning

In [ ]:
# Note: This is an intensive process requiring a GPU. 
# We just save the base model structure for the proxy version.
model_name = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

bert_tokenizer.save_pretrained('../model_pipeline/distilbert_model')
bert_model.save_pretrained('../model_pipeline/distilbert_model')
print("DistilBERT exported!")


## 6. Ensembling Configurations

In [ ]:
# Define weights for production
DEFAULT_WEIGHTS = {
    'distilbert': 0.5,
    'logistic': 0.2,
    'naive_bayes': 0.1,
    'random_forest': 0.1,
    'lstm': 0.1
}

save_ensemble_config('../model_pipeline/ensemble_config.json', DEFAULT_WEIGHTS)
print("Configuration saved!")
